# Imports and Setup

In [12]:
import pandas as pd
import json
from pathlib import Path

### Setup paths

In [13]:

# =====================================================================
# 1. PATHS
# =====================================================================

# Assumes the Jupyter kernel is running from:
# root/scripts/nlp/analysis_scripts/
#
# Check Path.cwd() if the results directory cannot be found.
# Also needs to be changed when run as a script

root_path = Path.cwd().parents[2]

results_path = root_path / "results" / "nlp-luna-all-comparison-v2"
analysis_path = root_path / "results" / "nlp_analysis"

if not results_path.is_dir():
    raise FileNotFoundError(f"Results directory not found: {results_path}")

analysis_path.mkdir(parents=True, exist_ok=True)

### Load JSON Utility

In [ ]:
# =====================================================================
# 2. LOAD ALL JSONL FILES
# =====================================================================

def load_statements(results_path):
    """
    Load JSONL records recursively from the experimental results.

    Expected folder structure:

        results/
            experiment/
                config/
                    app/
                        run/
                            agent-vs-http-app/
                                logs/
                                    *.jsonl

    Each JSONL record becomes one DataFrame row.

    All original JSON fields are retained as columns, alongside
    metadata identifying their experimental origin.
    """

    rows = []

    for file_path in sorted(results_path.rglob("*.jsonl")):

        relative_path = file_path.relative_to(results_path)
        parts = relative_path.parts

        # Expected:
        # config / app / run / task / ... / filename.jsonl
        if len(parts) < 5 or not parts[2].isdigit():
            continue

        config, app, run = parts[:3]

        source_file = str(Path(*parts[3:]))

        with file_path.open("r", encoding="utf-8") as f:

            for line_number, line in enumerate(f, start=1):

                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(
                        f"Invalid JSON in {file_path}, line {line_number}"
                    ) from e

                if not isinstance(record, dict):
                    continue

                # Flatten the JSON record into DataFrame columns.
                # Prefixing metadata avoids overwriting original fields.
                rows.append({
                    **record,
                    "config": config,
                    **parse_config(config),
                    "app": app,
                    "run": int(run),
                    "source_file": source_file,
                    "line_number": line_number,
                })

    return pd.DataFrame(rows)

# Utilities to do groupings

In [21]:
def parse_config(config):
    """
    Expected naming:
    attacker-model-m-attacker_strategy-defender-model-m-defender_strategy

    Assumes strategy names do not contain hyphens.
    """

    attacker_part, defender_part = config.split("-m-", 1)

    attacker_strategy, defender_config = defender_part.split("-", 1)

    defender_model, defender_strategy = defender_config.rsplit("-m-", 1)

    return {
        "attacker_model": attacker_part,
        "attacker_strategy": attacker_strategy,
        "defender_model": defender_model,
        "defender_strategy": defender_strategy,
    }

In [15]:
# =====================================================================
# 3. PREPARE STATEMENT-LEVEL DATA
# =====================================================================

def prepare_statements(df):
    """
    Clean and chronologically order the individual log records.

    Creates:
        interaction_id       Unique experimental run
        statement_id         Unique log record
        statement_order      Chronological position within interaction
        actor_statement_order Position within actor's timeline
        actor                Attacker / defender / unknown
        duration_seconds     Execution time of individual record

    Important:
        The actor is inferred from the source filename.
        Unknown filenames remain unclassified.
    """

    df = df.copy()

    if df.empty:
        raise ValueError("No JSONL records found.")

    # -------------------------------------------------------------
    # Convert timestamps and token counts
    # -------------------------------------------------------------

    numeric_columns = [
        "time_in",
        "time_out",
        "tokens_consumed",
        "tokens_generated",
        "cached_tokens",
        "cache_write_tokens",
    ]

    for col in numeric_columns:
        if col not in df.columns:
            df[col] = pd.NA

        df[col] = pd.to_numeric(df[col], errors="coerce")

    # -------------------------------------------------------------
    # Identify actor from source filename
    # -------------------------------------------------------------

    def identify_actor(source_file):

        filename = Path(source_file).stem.lower()

        if "attacker" in filename or "pentest" in filename:
            return "attacker"

        if "defender" in filename or "honeypot" in filename:
            return "defender"

        return "unknown"

    df["actor"] = df["source_file"].apply(identify_actor)

    # -------------------------------------------------------------
    # Construct interaction ID
    # -------------------------------------------------------------

    # Each configuration × application × run is one interaction.
    df["interaction_id"] = (
        df["config"].astype(str) + "|" +
        df["app"].astype(str) + "|" +
        df["run"].astype(str)
    )

    # -------------------------------------------------------------
    # Chronological sorting
    # -------------------------------------------------------------

    # time_in provides the primary chronological ordering.
    #
    # source_file and line_number are deterministic tie-breakers.
    # They do not establish causal relationships across log files.

    df = df.sort_values(
        [
            "config",
            "app",
            "run",
            "time_in",
            "time_out",
            "source_file",
            "line_number",
        ],
        kind="stable",
        na_position="last",
    ).reset_index(drop=True)

    # -------------------------------------------------------------
    # Statement identifiers
    # -------------------------------------------------------------

    # Unique within the entire dataset, including multiple log files.
    df["statement_id"] = (
        df["interaction_id"] + "|" +
        df["source_file"] + "|" +
        df["line_number"].astype(str)
    )

    # Position in the combined interaction timeline.
    df["statement_order"] = (
        df.groupby("interaction_id").cumcount() + 1
    )

    # Position in the actor-specific timeline.
    df["actor_statement_order"] = (
        df.groupby(["interaction_id", "actor"]).cumcount() + 1
    )

    # -------------------------------------------------------------
    # Duration
    # -------------------------------------------------------------

    df["duration_seconds"] = (
        df["time_out"] - df["time_in"]
    )

    return df


### Utility to generate exchange level data

In [16]:
# =====================================================================
# 4. GENERATE EXCHANGE-LEVEL DATA
# =====================================================================

def prepare_exchanges(df_statements):
    """
    Generate an initial exchange-level table.

    IMPORTANT:
        Actual attacker-defender communication boundaries have not
        yet been verified from HoneyVal's logging structure.

        We therefore assign one provisional exchange per statement.

        This guarantees a valid, lossless relationship between
        df_statements and df_exchanges, without inventing
        attacker-defender causal links.

    Later:
        Once exchange boundaries are identified, multiple statements
        can share the same exchange_id.
    """

    df = df_statements.copy()

    # Each statement is provisionally its own exchange.
    df["exchange_id"] = df["statement_id"]

    # Exchange order within each experimental interaction.
    df["exchange_order"] = df["statement_order"]

    exchanges = df[
        [
            "exchange_id",
            "interaction_id",
            "exchange_order",
            "actor",
            "time_in",
            "time_out",
            "statement_id",
        ]
    ].copy()

    exchanges = exchanges.rename(columns={
        "actor": "initiating_actor",
        "time_in": "start_time",
        "time_out": "end_time",
    })

    exchanges["n_statements"] = 1

    exchanges["duration_seconds"] = (
        exchanges["end_time"] - exchanges["start_time"]
    )

    return exchanges

### Utility to generate interaction level data

In [17]:
# =====================================================================
# 5. GENERATE INTERACTION-LEVEL DATA
# =====================================================================

def prepare_interactions(df_statements, df_exchanges):
    """
    Aggregate statement records into complete experimental runs.

    One row per:
        prompt configuration × application × run

    Provides:
        - Number of statements
        - Number of provisional exchanges
        - Start/end timestamps
        - Total interaction duration
        - Token consumption and generation

    Note:
        The resulting token sums cover the JSONL records loaded.
        They should not automatically be interpreted as complete
        experimental costs without checking HoneyVal's metrics.
    """

    df = df_statements.copy()

    interactions = (
        df.groupby(
            ["interaction_id", "config", "app", "run"],
            as_index=False
        )
        .agg(
            n_statements=("statement_id", "nunique"),
            start_time=("time_in", "min"),
            end_time=("time_out", "max"),
            total_tokens_consumed=("tokens_consumed", "sum"),
            total_tokens_generated=("tokens_generated", "sum"),
        )
    )

    # Number of exchanges belonging to each interaction.
    exchange_counts = (
        df_exchanges
        .groupby("interaction_id")
        .size()
        .rename("n_exchanges")
        .reset_index()
    )

    interactions = interactions.merge(
        exchange_counts,
        on="interaction_id",
        how="left",
        validate="one_to_one",
    )

    interactions["duration_seconds"] = (
        interactions["end_time"] -
        interactions["start_time"]
    )

    return interactions

### Generate all three levels of dataframes

In [18]:
# =====================================================================
# 6. GENERATE ALL THREE DATAFRAMES
# =====================================================================

df_statements = prepare_statements(
    load_statements(results_path)
)

df_exchanges = prepare_exchanges(
    df_statements
)

df_interactions = prepare_interactions(
    df_statements,
    df_exchanges
)

# Add provisional exchange identifiers to the statements table.
# This makes it possible to join statements and exchanges directly.
df_statements["exchange_id"] = df_statements["statement_id"]


### Assertions to validate if it went well

In [19]:
# =====================================================================
# 7. BASIC VALIDATION
# =====================================================================

# Each statement belongs to exactly one interaction.
assert df_statements["interaction_id"].notna().all()

# Statement IDs must be unique.
assert df_statements["statement_id"].is_unique

# Interaction IDs must be unique at interaction level.
assert df_interactions["interaction_id"].is_unique

# Exchange IDs must be unique at exchange level.
assert df_exchanges["exchange_id"].is_unique

# Every statement must belong to an existing interaction.
assert df_statements["interaction_id"].isin(
    df_interactions["interaction_id"]
).all()

# Every statement must belong to an existing exchange.
assert df_statements["exchange_id"].isin(
    df_exchanges["exchange_id"]
).all()


### Overview of resulting dataframes

In [20]:
# =====================================================================
# 8. OVERVIEW
# =====================================================================

print("=" * 60)
print("HONEYVAL DATASET")
print("=" * 60)

print(f"Configurations: {df_statements['config'].nunique():,}")
print(f"Applications:   {df_statements['app'].nunique():,}")
print(f"Interactions:   {len(df_interactions):,}")
print(f"Exchanges:      {len(df_exchanges):,} (provisional)")
print(f"Statements:     {len(df_statements):,}")

print("\nActor classification:")
print(df_statements["actor"].value_counts(dropna=False))

print("\nSource files:")
display(
    df_statements.groupby(["source_file", "actor"])
    .size()
    .reset_index(name="n_statements")
    .sort_values("source_file")
)

print("\nStatements:")
display(df_statements.head())

print("\nExchanges:")
display(df_exchanges.head())

print("\nInteractions:")
display(df_interactions.head())


HONEYVAL DATASET
Configurations: 2
Applications:   16
Interactions:   320
Exchanges:      6,775 (provisional)
Statements:     6,775

Actor classification:
actor
unknown    6775
Name: count, dtype: int64

Source files:


,source_file,actor,n_statements
0,agent-vs-http-app/metadata-logs/agent-metadata...,unknown,3620
1,agent-vs-http-app/metadata-logs/http-app-metad...,unknown,3155



Statements:


,time_in,time_out,event,raw_response,strategy,strategy_valid,strategy_source,strategy_attempts,strategy_prompt_version,strategy_retried,...,line_number,in,out,actor,interaction_id,statement_id,statement_order,actor_statement_order,duration_seconds,exchange_id
0,1.792772e+09,1.792772e+09,START,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NONE,NONE,unknown,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,1,0.000000,openai-gpt-6-luna-python-curl-m-exploit-openai...
1,1.792772e+09,1.792772e+09,TOOL,<ACTION>\n<ACTION_NAME>\nTOOL\n</ACTION_NAME>\...,The OpenAPI specification exposes a single JSO...,True,contemporaneous,1.0,strategic-rationale-v2,False,...,1,NaN,NaN,unknown,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,2,2,9.058358,openai-gpt-6-luna-python-curl-m-exploit-openai...
2,1.792772e+09,1.792772e+09,COMMAND,"<RESPONSE>\n<CONTENT>{""result"":""7""}</CONTENT>\...","The request submits the expression `1 + 2*3`, ...",True,contemporaneous,1.0,strategic-rationale-v2,False,...,2,"{'endpoint': '/calculator', 'method': 'POST', ...","<CONTENT>{""result"":""7""}</CONTENT>\n<MEDIA_TYPE...",unknown,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,3,3,2.885659,openai-gpt-6-luna-python-curl-m-exploit-openai...
3,1.792772e+09,1.792772e+09,PARSING ERROR,<ACTION>\n<ACTION_NAME>\nTOOL\n</ACTION_NAME>\...,The calculator returned `7` for a normal arith...,True,contemporaneous,1.0,strategic-rationale-v2,False,...,2,NaN,NaN,unknown,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,4,4,5.789832,openai-gpt-6-luna-python-curl-m-exploit-openai...
4,1.792772e+09,1.792772e+09,TOOL,<ACTION>\n<ACTION_NAME>\nTOOL\n</ACTION_NAME>\...,The calculator returns a normal result for bas...,True,contemporaneous,1.0,strategic-rationale-v2,False,...,3,NaN,NaN,unknown,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,5,5,11.327734,openai-gpt-6-luna-python-curl-m-exploit-openai...



Exchanges:


,exchange_id,interaction_id,exchange_order,initiating_actor,start_time,end_time,statement_id,n_statements,duration_seconds
0,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,unknown,1.792772e+09,1.792772e+09,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,0.000000
1,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,2,unknown,1.792772e+09,1.792772e+09,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,9.058358
2,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,3,unknown,1.792772e+09,1.792772e+09,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,2.885659
3,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,4,unknown,1.792772e+09,1.792772e+09,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,5.789832
4,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,5,unknown,1.792772e+09,1.792772e+09,openai-gpt-6-luna-python-curl-m-exploit-openai...,1,11.327734



Interactions:


,interaction_id,config,app,run,n_statements,start_time,end_time,total_tokens_consumed,total_tokens_generated,n_exchanges,duration_seconds
0,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,Calculator,0,19,1.792772e+09,1.792772e+09,36459,8399,19,120.605144
1,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,Calculator,1,18,1.792772e+09,1.792772e+09,35003,9217,18,189.638176
2,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,Calculator,2,18,1.792772e+09,1.792772e+09,32410,8059,18,128.965794
3,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,Calculator,3,19,1.792772e+09,1.792772e+09,34985,9223,19,138.155881
4,openai-gpt-6-luna-python-curl-m-exploit-openai...,openai-gpt-6-luna-python-curl-m-exploit-openai...,Calculator,4,13,1.792772e+09,1.792772e+09,18497,5223,13,92.137191


### Saving the dataframes

In [ ]:
# =====================================================================
# 9. SAVE DATAFRAMES
# =====================================================================

# Convert nested JSON structures into JSON strings for reliable
# Parquet serialization. This is only applied to saved copies.
def prepare_for_parquet(df):
    output = df.copy()

    for col in output.columns:
        if output[col].map(
            lambda x: isinstance(x, (dict, list))
        ).any():
            output[col] = output[col].map(
                lambda x: json.dumps(x, ensure_ascii=False)
                if isinstance(x, (dict, list)) else x
            )

    return output


prepare_for_parquet(df_statements).to_parquet(
    analysis_path / "statements.parquet",
    index=False
)

prepare_for_parquet(df_exchanges).to_parquet(
    analysis_path / "exchanges.parquet",
    index=False
)

prepare_for_parquet(df_interactions).to_parquet(
    analysis_path / "interactions.parquet",
    index=False
)

print(f"\nSaved analysis data to: {analysis_path}")